## 1. Import thư viện

In [21]:
import json
import ijson
import faiss
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

## 2. Config

Khai báo file chunks, thư mục lưu embedding/index đầu ra và model E5 sẽ dùng. `OUTPUT_DIR.mkdir(...)` đảm bảo thư mục kết quả tồn tại trước khi ghi file.

In [ ]:
CHUNKS_PATH = Path("data/corpus_chunks.json")
OUTPUT_DIR = Path("data/e5_base")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "intfloat/multilingual-e5-base" 

### Cấu hình batch và kích thước xử lý

Cell này đặt các tham số chạy embedding: số mẫu mỗi batch, độ dài token tối đa, số vector mỗi shard, giới hạn bản ghi khi thử nghiệm và cột text dùng để encode.

In [23]:
BATCH_SIZE = 16
MAX_LENGTH = 512
SHARD_SIZE = 10_000
MAX_RECORDS = None
TEXT_COL = "raw_text"

In [24]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.7.1+cu118
CUDA available: True


## 3. Load Model

### Load tokenizer và model E5

Cell này chọn GPU nếu có, dùng `float16` trên CUDA để tiết kiệm bộ nhớ, sau đó load tokenizer/model `multilingual-e5-base` và đưa model về chế độ evaluation.

In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)

model.eval()

device: cuda
dtype: torch.float16


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1275.22it/s]


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

## 4. Hàm encode chuẩn cho E5

### Mean pooling token embeddings

Cell này định nghĩa cách gom embedding của từng token thành một vector đại diện cho cả đoạn text. `attention_mask` giúp bỏ qua padding token khi tính trung bình.

In [26]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

### Hàm encode danh sách văn bản

Cell này biến danh sách text thành ma trận embedding. Mỗi batch sẽ được tokenize, đưa qua model, mean pooling, normalize L2 và trả về `float32` để FAISS có thể index/search ổn định.

In [27]:
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(
            embeddings.cpu().numpy().astype("float32")
        )

        del inputs, outputs, embeddings

        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

### Đọc file chunks dạng JSON array

Cell này tạo generator đọc từng item trong file JSON lớn bằng `ijson`. Cách này tránh load toàn bộ corpus vào RAM cùng lúc.

In [28]:
def iter_chunks_json_array(path):
    with open(path, "rb") as f:
        for item in ijson.items(f, "item"):
            yield item

### Lưu embedding shard và metadata

Cell này lưu vector của một shard vào `.npy` và metadata tương ứng vào `.parquet`. Hai file có cùng `shard_id` để sau này ghép lại đúng thứ tự.

In [29]:
def save_shard(shard_id, embeddings, metadata_rows):
    emb_path = OUTPUT_DIR / f"embeddings_part_{shard_id:05d}.npy"
    meta_path = OUTPUT_DIR / f"metadata_part_{shard_id:05d}.parquet"
    embeddings = np.asarray(embeddings, dtype="float32")
    metadata=pd.DataFrame(metadata_rows)

    np.save(emb_path,embeddings)
    metadata.to_parquet(meta_path, index=False)

    print("saved:", emb_path, embeddings.shape)
    print("saved:", meta_path, metadata.shape)

### Tạo embedding cho corpus

Cell này lặp qua các chunk, thêm prefix `passage: ` theo chuẩn E5, encode theo batch, gom thành shard và lưu ra disk. `MAX_RECORDS` đang dùng để giới hạn khi chạy thử; đặt `None` nếu muốn encode toàn bộ corpus.

In [30]:
batch_texts = []
batch_meta = []

shard_embeddings = []
shard_metadata = []

shard_id = 0
total_records = 0
total_embedded = 0

for item in tqdm(iter_chunks_json_array(CHUNKS_PATH)):
    if MAX_RECORDS is not None and total_records >= MAX_RECORDS:
        break

    total_records += 1

    text = str(item.get(TEXT_COL, "")).strip()

    if not text:
        continue

    batch_texts.append("passage: " + text)
    batch_meta.append({
        "doc_id": item.get("doc_id"),
        "chunk_id": item.get("chunk_id"),
        "url": item.get("url"),
        "topic": item.get("topic"),
        "text": text,
    })

    if len(batch_texts) >= BATCH_SIZE :
        embeddings = encode_texts(batch_texts)
        shard_embeddings.append(embeddings)
        shard_metadata.extend(batch_meta)

        total_embedded += len(batch_texts)
        batch_texts = []
        batch_meta = []

        if len(shard_metadata) >= SHARD_SIZE:
            shard_embeddings = np.vstack(shard_embeddings)

            save_shard(
                shard_id=shard_id,
                embeddings=shard_embeddings,
                metadata_rows=shard_metadata,
            )

            shard_id += 1
            shard_embeddings = []
            shard_metadata = []

if batch_texts:
    embeddings = encode_texts(batch_texts)

    shard_embeddings.append(embeddings)
    shard_metadata.extend(batch_meta)

    total_embedded += len(batch_texts)

if shard_metadata:
    shard_embeddings = np.vstack(shard_embeddings)

    save_shard(
        shard_id=shard_id,
        embeddings=shard_embeddings,
        metadata_rows=shard_metadata,
    )

print("total_records:", total_records)
print("total_embedded:", total_embedded)

10032it [01:03, 111.90it/s]

saved: data\e5_base\embeddings_part_00000.npy (10000, 768)
saved: data\e5_base\metadata_part_00000.parquet (10000, 5)


20016it [02:02, 125.86it/s]

saved: data\e5_base\embeddings_part_00001.npy (10000, 768)
saved: data\e5_base\metadata_part_00001.parquet (10000, 5)


30004it [03:01, 156.89it/s]

...

saved: data\e5_base\embeddings_part_00105.npy (10000, 768)
saved: data\e5_base\metadata_part_00105.parquet (10000, 5)


1070032it [1:47:31, 163.30it/s]

saved: data\e5_base\embeddings_part_00106.npy (10000, 768)
saved: data\e5_base\metadata_part_00106.parquet (10000, 5)


1078604it [1:48:17, 166.00it/s]

saved: data\e5_base\embeddings_part_00107.npy (8604, 768)
saved: data\e5_base\metadata_part_00107.parquet (8604, 5)
total_records: 1078604
total_embedded: 1078604


## 5. Build FAISS Index

### Kiểm tra các shard đã tạo

Cell này tìm tất cả file embedding và metadata đã lưu trong `OUTPUT_DIR`, sau đó in số shard để đảm bảo dữ liệu đầu vào cho bước build FAISS là đầy đủ.

In [31]:
embedding_files = sorted(OUTPUT_DIR.glob("embeddings_part_*.npy"))
metadata_files = sorted(OUTPUT_DIR.glob("metadata_part_*.parquet"))

print("embedding shards:", len(embedding_files))
print("metadata shards:", len(metadata_files))

embedding shards: 108
metadata shards: 108


### Build FAISS index từ embedding shards

Cell này đọc từng shard embedding, tạo `IndexFlatIP`, add vector vào index và ghép metadata lại thành một bảng duy nhất. Vì vector đã normalize nên inner product tương đương cosine similarity.

In [32]:
index = None
all_metadata = []
offset = 0

for emb_file, meta_file in tqdm(list(zip(embedding_files, metadata_files))):
    embeddings = np.load(emb_file).astype("float32")
    metadata = pd.read_parquet(meta_file)

    if index is None:
        dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)

    metadata.insert(
        0,
        "embedding_index",
        np.arange(offset, offset+len(metadata))
    )
    index.add(embeddings)
    all_metadata.append(metadata)

    offset += len(metadata)

metadata = pd.concat(all_metadata, ignore_index=True)
print("faiss vectors:", index.ntotal)
print("metadata rows:", len(metadata))

100%|██████████| 108/108 [00:12<00:00,  8.34it/s]

faiss vectors: 1078604
metadata rows: 1078604


### Khai báo đường dẫn artifact semantic

Cell này đặt tên file cho FAISS index, metadata tổng hợp và config embedding. Các artifact này sẽ được load lại khi search mà không cần encode corpus lại.

In [33]:
INDEX_PATH = OUTPUT_DIR / "semantic_e5_base.faiss"
METADATA_PATH = OUTPUT_DIR / "chunks_metadata.parquet"
CONFIG_PATH = OUTPUT_DIR / "embedding_config.json"

### Ghi index và metadata ra disk

Cell này lưu FAISS index vào `.faiss` và metadata vào `.parquet`. Đây là kết quả chính của bước indexing semantic.

In [34]:
faiss.write_index(index, str(INDEX_PATH))
metadata.to_parquet(METADATA_PATH, index=False)

### Lưu config embedding

Cell này ghi lại các thông tin quan trọng của pipeline: model, prefix document/query, max length, metric FAISS, kích thước vector và số vector. Config này giúp search về sau dùng đúng cách encode ban đầu.

In [35]:
config = {
    "model_name": MODEL_NAME,
    "text_column": TEXT_COL,
    "document_prefix": "passage: ",
    "query_prefix": "query: ",
    "max_length": MAX_LENGTH,
    "normalize_embeddings": True,
    "faiss_metric": "inner_product",
    "embedding_dim": int(index.d),
    "num_vectors": int(index.ntotal),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

config

{'model_name': 'intfloat/multilingual-e5-base',
 'text_column': 'raw_text',
 'document_prefix': 'passage: ',
 'query_prefix': 'query: ',
 'max_length': 512,
 'normalize_embeddings': True,
 'faiss_metric': 'inner_product',
 'embedding_dim': 768,
 'num_vectors': 1078604}